# LangChain Summarization Middleware — compaction vs rule adherence

Benchmarks LangChain's `SummarizationMiddleware` on a long-horizon research agent that must obey formatting rules seeded early in the conversation.

Two questions per model × token threshold:

1. **How often did context get compacted?** Counted via a wrapper around the summarizer LLM (one `ainvoke` == one compaction), cross-checked against summary messages in the final state.
2. **Did the final one-pager still obey the rules** after early rule-bearing turns were summarized away?

Each run seeds scripted user/assistant turns that restate formatting rules, then sends the research task. With `keep=("messages", 4)` and ~10–15 search/fetch tool turns, early rule messages are dropped when `trigger=("tokens", N)` fires. The `None` threshold run is the no-compaction control.

**Prereqs:** `FIREWORKS_API_KEY`, `ANTHROPIC_API_KEY`, and `SERP_API_KEY` in `training/.env` or your shell.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q langchain langchain-core langchain-fireworks langchain-anthropic google-search-results trafilatura matplotlib pandas pydantic httpx python-dotenv

In [2]:
# --- edit these ---
MAIN_MODELS: dict[str, tuple[str, str]] = {
    "GLM-5.2 (Fireworks)": ("fireworks", "accounts/fireworks/models/glm-5p2"),
    "Opus 4.8 (Anthropic)": ("anthropic", "claude-opus-4-8"),
}

THRESHOLDS: list[int | None] = [1500, 3000, 6000, None]  # None = no summarization baseline
KEEP: tuple[str, int] = ("messages", 4)

RESEARCH_TOPIC = (
    "The state of small modular reactors (SMRs) in 2026: "
    "technology, economics, regulation, and notable projects"
)

MAX_STEPS = 200  # recursion_limit for agent.ainvoke

# Incremental save: each completed run is one JSONL line + fsync.
# RESUME=True skips (model, threshold) pairs already in the file.
SAVE_PATH = "langchain_compaction_runs.jsonl"
RESUME = False

# Run to inspect in the final cell (0-indexed into `runs`).
INSPECT_RUN = 0

In [3]:
import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Any

import httpx
import matplotlib.pyplot as plt
import pandas as pd
import trafilatura
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, messages_to_dict
from langchain_core.tools import tool
from langchain_fireworks import ChatFireworks
from pydantic import BaseModel, Field
from serpapi import GoogleSearch

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")

missing = [k for k in ("FIREWORKS_API_KEY", "ANTHROPIC_API_KEY", "SERP_API_KEY") if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing {missing}. Set them in {training_dir / '.env'} or your shell.")

# Held constant across all runs — only main model + threshold vary.
SUMMARIZER = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# Self-judge caveat: when Opus 4.8 is a subject model, the judge is the same family.
# Swap to a different judge model if you want blind grading.
JUDGE = ChatAnthropic(model="claude-opus-4-8", temperature=0)

print("Setup complete.")

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## Rules, prompts, and seeded conversation

Rules live in the system prompt **and** in scripted early turns so they are compacted away once the token threshold is hit. `KEEP=("messages", 4)` keeps only the four most recent messages after each summary.

In [4]:
RULES: list[str] = [
    "Final output is a single cohesive one-pager (not a bulleted dump).",
    "Cites at least 8 distinct sources by index or URL.",
    "Under 500 words excluding citations.",
    'No first-person pronouns ("I", "we", "my", "our").',
    'Ends with a section titled exactly "Verdict".',
    'Includes a section titled exactly "Key Findings".',
    "No verbatim copy-paste of fetched source text.",
    "Neutral report tone.",
]

RULES_NUMBERED = "\n".join(f"{i + 1}. {r}" for i, r in enumerate(RULES))
RULES_REPHRASED = (
    "Deliver one cohesive narrative page with Key Findings and a Verdict section, "
    "cite eight or more distinct sources, stay under 500 words (excluding citations), "
    "avoid first-person pronouns, do not paste source text verbatim, and keep a neutral tone."
)

SYSTEM_PROMPT = f"""You are a research analyst writing a final one-pager.

Formatting rules for the final deliverable (must follow even after context is summarized):
{RULES_NUMBERED}

Workflow:
1. Use web_search and fetch_url to gather roughly 10–15 sources on the topic.
2. Read fetched pages; take notes internally.
3. Write the final one-pager obeying every rule above, then stop (no more tool calls).
"""

SUMMARY_MARKER = "Here is a summary of the conversation to date:"


def seed_messages(topic: str) -> list[dict[str, str]]:
    return [
        {
            "role": "user",
            "content": (
                "Before we start, here are the formatting rules for the final deliverable:\n"
                f"{RULES_NUMBERED}\nAcknowledge."
            ),
        },
        {
            "role": "assistant",
            "content": "Understood. I'll follow all formatting rules for the final one-pager.",
        },
        {
            "role": "user",
            "content": (
                f"To reiterate: {RULES_REPHRASED} "
                "Keep these even as the conversation grows long."
            ),
        },
        {
            "role": "assistant",
            "content": "Got it. I'll keep to these rules throughout.",
        },
        {
            "role": "user",
            "content": (
                f"Research topic: {topic}\n\n"
                "Gather approximately 10–15 sources with web_search and fetch_url, "
                "then write the final one-pager and stop."
            ),
        },
    ]

## Tools

`web_search` hits SerpAPI; `fetch_url` returns cleaned page text capped at ~3500 chars so each fetch is a bounded context contribution.

In [5]:
FETCH_CHAR_CAP = 3500


@tool
def web_search(query: str) -> list[dict[str, str]]:
    """Search the web and return up to 10 results with title, url, and snippet."""
    try:
        payload = GoogleSearch({
            "q": query,
            "api_key": os.environ["SERP_API_KEY"],
            "num": 10,
        }).get_dict()
        out: list[dict[str, str]] = []
        for hit in payload.get("organic_results", [])[:10]:
            out.append({
                "title": hit.get("title", ""),
                "url": hit.get("link", ""),
                "snippet": hit.get("snippet", ""),
            })
        return out
    except Exception as e:
        return [{"title": "ERROR", "url": "", "snippet": f"web_search failed: {e}"}]


@tool
def fetch_url(url: str) -> str:
    """Fetch a URL and return cleaned article text (first ~3500 chars)."""
    try:
        with httpx.Client(timeout=30.0, follow_redirects=True) as client:
            resp = client.get(url, headers={"User-Agent": "langchain-compaction-benchmark/1.0"})
            resp.raise_for_status()
            html = resp.text
        text = trafilatura.extract(html, url=url) or ""
        text = re.sub(r"\s+", " ", text).strip()
        if len(text) > FETCH_CHAR_CAP:
            text = text[:FETCH_CHAR_CAP] + "…"
        return text or "(no extractable text)"
    except Exception as e:
        return f"(error fetching url: {e})"


TOOLS = [web_search, fetch_url]

## Counting summarizer + agent builder

In [6]:
class CountingChatModel:
    """Thin wrapper: one summarizer invoke == one compaction event."""

    def __init__(self, inner: Any) -> None:
        self.inner = inner
        self.compaction_count = 0

    def reset(self) -> None:
        self.compaction_count = 0

    def invoke(self, *args: Any, **kwargs: Any) -> Any:
        self.compaction_count += 1
        return self.inner.invoke(*args, **kwargs)

    async def ainvoke(self, *args: Any, **kwargs: Any) -> Any:
        self.compaction_count += 1
        return await self.inner.ainvoke(*args, **kwargs)

    def bind_tools(self, *args: Any, **kwargs: Any) -> "CountingChatModel":
        wrapped = CountingChatModel(self.inner.bind_tools(*args, **kwargs))
        wrapped.compaction_count = self.compaction_count
        return wrapped

    def with_structured_output(self, *args: Any, **kwargs: Any) -> "CountingChatModel":
        wrapped = CountingChatModel(self.inner.with_structured_output(*args, **kwargs))
        wrapped.compaction_count = self.compaction_count
        return wrapped

    def __getattr__(self, name: str) -> Any:
        return getattr(self.inner, name)


counting_summarizer = CountingChatModel(SUMMARIZER)


def build_main_model(provider: str, model_id: str) -> ChatFireworks | ChatAnthropic:
    if provider == "fireworks":
        return ChatFireworks(model=model_id, temperature=0)
    if provider == "anthropic":
        # Opus 4.8 rejects temperature — omit it.
        return ChatAnthropic(model=model_id)
    raise ValueError(f"Unknown provider: {provider}")


def build_agent(main_model: ChatFireworks | ChatAnthropic, threshold: int | None):
    middleware = []
    if threshold is not None:
        middleware.append(
            SummarizationMiddleware(
                model=counting_summarizer,
                trigger=("tokens", threshold),
                keep=KEEP,
            )
        )
    return create_agent(
        model=main_model,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        middleware=middleware,
    )


def message_content(msg: BaseMessage | dict) -> str:
    if isinstance(msg, dict):
        content = msg.get("content", "")
    else:
        content = msg.content
    if isinstance(content, list):
        return " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content or "")


def count_summary_messages(messages: list) -> int:
    n = 0
    for m in messages:
        if SUMMARY_MARKER in message_content(m):
            n += 1
            continue
        if isinstance(m, HumanMessage) and m.additional_kwargs.get("lc_source") == "summarization":
            n += 1
    return n


def extract_final_one_pager(messages: list) -> str:
    for m in reversed(messages):
        if not isinstance(m, AIMessage):
            continue
        if getattr(m, "tool_calls", None):
            continue
        text = message_content(m).strip()
        if text:
            return text
    return ""

## Structured judge + deterministic pre-checks

In [7]:
class RuleResult(BaseModel):
    rule_id: int = Field(description="1-based rule index matching the RULES list")
    rule_text: str
    passed: bool
    reason: str


class JudgeResult(BaseModel):
    results: list[RuleResult]
    overall_score: float = Field(description="Fraction of rules passed (0-1)")


structured_judge = JUDGE.with_structured_output(JudgeResult)

_FIRST_PERSON_RE = re.compile(r"\b(I|we|my|our|mine|ours|us)\b", re.IGNORECASE)


def word_count_excl_citations(text: str) -> int:
    stripped = re.sub(r"https?://\S+", "", text)
    stripped = re.sub(r"\[\d+\]", "", stripped)
    return len(stripped.split())


def deterministic_checks(text: str) -> dict[int, dict[str, Any]]:
    """Cheap pre-checks for rules 3 (word count) and 4 (first person)."""
    wc = word_count_excl_citations(text)
    return {
        3: {"det_ok": wc <= 500, "word_count": wc},
        4: {"det_ok": _FIRST_PERSON_RE.search(text) is None},
    }


JUDGE_PROMPT = """You are grading a research one-pager against a fixed rule list.

Ground-truth rules (evaluate every rule):
{rules}

Candidate one-pager (only this text — not the conversation):
---
{one_pager}
---

For each rule, return rule_id (1-based), rule_text, passed (bool), and a short reason.
overall_score = fraction of rules passed.
"""


async def grade_one_pager(one_pager: str) -> tuple[JudgeResult, dict[int, dict[str, Any]]]:
    det = deterministic_checks(one_pager)
    prompt = JUDGE_PROMPT.format(rules=RULES_NUMBERED, one_pager=one_pager)
    result: JudgeResult = await structured_judge.ainvoke(prompt)
    return result, det

## Run loop

Iterates `MAIN_MODELS × THRESHOLDS`, appends each finished run to `SAVE_PATH`, and grades the final one-pager.

In [8]:
def run_key(model_name: str, threshold: int | None) -> str:
    return f"{model_name}::{threshold if threshold is not None else 'none'}"


def serialize_messages(messages: list) -> list[dict]:
    if messages and isinstance(messages[0], BaseMessage):
        return messages_to_dict(messages)
    return messages


async def run_single(model_name: str, provider: str, model_id: str, threshold: int | None) -> dict:
    counting_summarizer.reset()
    main_model = build_main_model(provider, model_id)
    agent = build_agent(main_model, threshold)

    print(f"\n--- {model_name} @ threshold={threshold!r} ---", flush=True)
    t0 = time.perf_counter()
    state = await agent.ainvoke(
        {"messages": seed_messages(RESEARCH_TOPIC)},
        config={"recursion_limit": MAX_STEPS},
    )
    wall_s = time.perf_counter() - t0

    messages = state["messages"]
    final_text = extract_final_one_pager(messages)
    compactions_counter = counting_summarizer.compaction_count
    compactions_messages = count_summary_messages(messages)
    mismatch = compactions_counter != compactions_messages
    if mismatch:
        print(
            f"  WARNING: compaction count mismatch "
            f"(counter={compactions_counter}, messages={compactions_messages})",
            flush=True,
        )

    judge_result, det_checks = await grade_one_pager(final_text)
    wc = word_count_excl_citations(final_text)

    summary_msgs = [
        message_content(m)
        for m in messages
        if SUMMARY_MARKER in message_content(m)
        or (isinstance(m, HumanMessage) and m.additional_kwargs.get("lc_source") == "summarization")
    ]

    row = {
        "run_key": run_key(model_name, threshold),
        "model": model_name,
        "threshold": threshold,
        "compactions": compactions_counter,
        "compactions_messages": compactions_messages,
        "compaction_mismatch": mismatch,
        "overall_score": judge_result.overall_score,
        "rule_results": [r.model_dump() for r in judge_result.results],
        "det_checks": {str(k): v for k, v in det_checks.items()},
        "word_count": wc,
        "wall_time_s": round(wall_s, 2),
        "final_text": final_text,
        "seeded_messages": seed_messages(RESEARCH_TOPIC),
        "summary_messages": summary_msgs,
        "all_messages": serialize_messages(messages),
    }
    print(
        f"  compactions={compactions_counter}, score={judge_result.overall_score:.2f}, "
        f"words={wc}, wall={wall_s:.1f}s",
        flush=True,
    )
    return row


async def run_benchmark() -> list[dict]:
    runs: list[dict] = []
    done_keys: set[str] = set()

    if RESUME and SAVE_PATH and os.path.exists(SAVE_PATH):
        with open(SAVE_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                runs.append(row)
                done_keys.add(row["run_key"])
        print(f"Resumed {len(runs)} run(s) from {SAVE_PATH}: {sorted(done_keys)}", flush=True)

    save_fh = open(SAVE_PATH, "a") if SAVE_PATH else None
    try:
        for model_name, (provider, model_id) in MAIN_MODELS.items():
            for threshold in THRESHOLDS:
                key = run_key(model_name, threshold)
                if key in done_keys:
                    print(f"Skipping {key} (already saved)", flush=True)
                    continue
                row = await run_single(model_name, provider, model_id, threshold)
                runs.append(row)
                if save_fh is not None:
                    save_fh.write(json.dumps(row) + "\n")
                    save_fh.flush()
                    os.fsync(save_fh.fileno())
                    print(f"  [saved {key} to {SAVE_PATH}]", flush=True)
    finally:
        if save_fh is not None:
            save_fh.close()
    return runs


runs = await run_benchmark()
print(f"\nBenchmark complete: {len(runs)} run(s).")


--- GLM-5.2 (Fireworks) @ threshold=1500 ---


Retrying langchain_fireworks.chat_models._acompletion_with_retry.<locals>._call in 4 seconds as it raised APITimeoutError: Request timed out..


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CcNZM4tyrjoG6VvSvrV9m'}

### Recover from a partial run

If the kernel died mid-benchmark, reload finished runs from `SAVE_PATH` so the results cells below work without re-running.

In [ ]:
if SAVE_PATH and os.path.exists(SAVE_PATH):
    loaded = []
    with open(SAVE_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                loaded.append(json.loads(line))
    print(f"Loaded {len(loaded)} run(s) from {SAVE_PATH}. Setting `runs = loaded`.")
    runs = loaded
else:
    print(f"No save file at {SAVE_PATH!r}; keeping in-memory `runs` ({len(runs)} run(s)).")

## Results

In [ ]:
def flatten_run(row: dict) -> dict:
    flat = {
        "model": row["model"],
        "threshold": row["threshold"],
        "compactions": row["compactions"],
        "compactions_messages": row["compactions_messages"],
        "compaction_mismatch": row["compaction_mismatch"],
        "overall_score": row["overall_score"],
        "word_count": row["word_count"],
        "wall_time_s": row["wall_time_s"],
    }
    for rr in row.get("rule_results", []):
        flat[f"rule_{rr['rule_id']}_pass"] = rr["passed"]
    det = row.get("det_checks", {})
    if "3" in det:
        flat["det_rule3_wordcount_ok"] = det["3"].get("det_ok")
    if "4" in det:
        flat["det_rule4_firstperson_ok"] = det["4"].get("det_ok")
    return flat


df = pd.DataFrame([flatten_run(r) for r in runs])
df_display = df.copy()
df_display["threshold"] = df_display["threshold"].apply(lambda t: "none" if t is None else t)
df_display

In [ ]:
def threshold_label(t: int | None) -> str:
    return "none" if t is None else str(t)


def threshold_sort_key(t: int | None) -> float:
    return -1 if t is None else float(t)


plot_df = df.copy()
plot_df["threshold_label"] = plot_df["threshold"].apply(threshold_label)
plot_df["threshold_sort"] = plot_df["threshold"].apply(threshold_sort_key)

models = list(MAIN_MODELS.keys())
colors = plt.cm.tab10.colors
thresholds_sorted = sorted(THRESHOLDS, key=threshold_sort_key)
x_labels = [threshold_label(t) for t in thresholds_sorted]
x_pos = list(range(len(x_labels)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("LangChain SummarizationMiddleware — compaction vs rule adherence", fontsize=13)

# 1. Compactions vs threshold
ax = axes[0]
for i, model in enumerate(models):
    sub = plot_df[plot_df["model"] == model].sort_values("threshold_sort")
    ys = [sub[sub["threshold"] == t]["compactions"].iloc[0] if len(sub[sub["threshold"] == t]) else 0 for t in thresholds_sorted]
    ax.plot(x_pos, ys, marker="o", label=model, color=colors[i % len(colors)])
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.set_xlabel("token threshold")
ax.set_ylabel("compactions")
ax.set_title("Compactions vs threshold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Rule adherence vs threshold
ax = axes[1]
for i, model in enumerate(models):
    sub = plot_df[plot_df["model"] == model].sort_values("threshold_sort")
    ys = [sub[sub["threshold"] == t]["overall_score"].iloc[0] if len(sub[sub["threshold"] == t]) else float("nan") for t in thresholds_sorted]
    ax.plot(x_pos, ys, marker="o", label=model, color=colors[i % len(colors)])
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.set_xlabel("token threshold")
ax.set_ylabel("overall_score")
ax.set_ylim(0, 1.05)
ax.set_title("Rule adherence vs threshold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig("langchain_compaction_rules.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved langchain_compaction_rules.png")

In [ ]:
# Per-rule compliance heatmap (rules × model+threshold)
rule_cols = sorted(c for c in df.columns if c.startswith("rule_") and c.endswith("_pass"))
if rule_cols and len(runs):
    heat_rows = []
    row_labels = []
    for r in runs:
        label = f"{r['model']}\n{t if (t := r['threshold']) is not None else 'none'}"
        row_labels.append(label)
        flat = flatten_run(r)
        heat_rows.append([1.0 if flat.get(c) else 0.0 for c in rule_cols])

    heat = pd.DataFrame(heat_rows, index=row_labels, columns=[c.replace("rule_", "R").replace("_pass", "") for c in rule_cols])

    fig, ax = plt.subplots(figsize=(max(8, len(rule_cols) * 0.9), max(3, len(row_labels) * 0.55)))
    im = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=8)
    ax.set_title("Per-rule compliance (green=pass)")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, "✓" if heat.values[i, j] else "✗", ha="center", va="center", color="black", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.02)
    plt.tight_layout()
    plt.savefig("langchain_compaction_rules_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved langchain_compaction_rules_heatmap.png")
else:
    print("No rule columns to plot yet — run the benchmark first.")

## Inspect one run

Seeded conversation, compaction summaries, and the final one-pager for `INSPECT_RUN`.

In [ ]:
r = runs[INSPECT_RUN]
print(f"Run: {r['run_key']}")
print(f"Compactions: counter={r['compactions']}, messages={r['compactions_messages']}, mismatch={r['compaction_mismatch']}")
print(f"Overall score: {r['overall_score']:.2f} | word_count={r['word_count']} | wall={r['wall_time_s']}s")
print(f"Det checks: {r.get('det_checks')}")

print("\n=== Seeded conversation ===")
for m in r.get("seeded_messages", []):
    print(f"\n[{m['role'].upper()}]\n{m['content'][:500]}{'…' if len(m['content']) > 500 else ''}")

print("\n=== Compaction summary messages ===")
for i, sm in enumerate(r.get("summary_messages", []), 1):
    print(f"\n--- summary {i} ---\n{sm[:1200]}{'…' if len(sm) > 1200 else ''}")
if not r.get("summary_messages"):
    print("(none — threshold was None or compaction never triggered)")

print("\n=== Final one-pager ===")
print(r.get("final_text", "(empty)"))

print("\n=== Per-rule judge ===")
for rr in r.get("rule_results", []):
    mark = "PASS" if rr["passed"] else "FAIL"
    print(f"  R{rr['rule_id']}: {mark} — {rr['reason']}")

### Notes

- **Token trigger is approximate** — `SummarizationMiddleware` uses `count_tokens_approximately` by default. Compaction counts may not line up exactly with provider token meters.
- **Self-judge caveat** — when Opus 4.8 is a subject model, `JUDGE` is the same model family. Swap `JUDGE` in the setup cell for blind grading.
- **Anthropic temperature** — Opus 4.8 rejects `temperature`; Fireworks GLM-5.2 uses `temperature=0`.
- **Incremental save / resume** — each finished run appends to `SAVE_PATH` and is `fsync`ed. Set `RESUME=True` to skip completed `(model, threshold)` pairs. Delete or rename the save file for a fresh run.
- **SERP_API_KEY** — required for `web_search`; add it to `training/.env` before running.